# Customer Churn Prediction: Profit-Based Evaluation of ML Models
## Step 7: Profit-Based Evaluation — Cost Matrix & Expected Maximum Profit

In [ ]:
# ── IMPORTS ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import joblib

from sklearn.metrics import confusion_matrix

sns.set_theme(style='whitegrid')

# Load data
X_test  = pd.read_csv('X_test.csv')
y_test  = pd.read_csv('y_test.csv').squeeze()

# Load trained models
models = {
    'Logistic Regression' : joblib.load('logistic_regression_model.pkl'),
    'Random Forest'       : joblib.load('random_forest_model.pkl'),
    'Gradient Boosting'   : joblib.load('gradient_boosting_model.pkl')
}

# Load traditional rankings for final comparison
traditional_metrics  = pd.read_csv('traditional_metrics.csv', index_col='Model')
traditional_rankings = pd.read_csv('traditional_rankings.csv', index_col='Model')

print('Data, models and previous results loaded ✓')

### 7.1 — Define the Cost-Profit Matrix

Since real customer profit data is unavailable, we use cost assumptions from churn prediction literature.

| Prediction | Actual | Outcome | Financial Impact |
|---|---|---|---|
| Churn (1) | Churn (1) | **True Positive** | Retention offer sent → customer saved. Benefit minus offer cost. |
| No Churn (0) | Churn (1) | **False Negative** | Churner missed → customer lost. Full revenue loss. |
| Churn (1) | No Churn (0) | **False Positive** | Unnecessary offer sent → wasted retention cost. |
| No Churn (0) | No Churn (0) | **True Negative** | Correctly ignored → no cost, no action needed. |

In [ ]:
# ── COST-PROFIT MATRIX VALUES ─────────────────────────────────────────────
# Based on literature assumptions (Verbraken et al., 2013 — EMP framework)
# Units: $ per customer

# Average Monthly Charge from the dataset (computed in EDA)
# Used as a proxy for customer value
avg_monthly_charge = 64.76   # from EDA Step 3
clv_months         = 12      # assume 12-month customer lifetime value window
CLV                = avg_monthly_charge * clv_months   # ~$777

# Retention offer cost (discount/incentive to keep customer)
retention_cost = avg_monthly_charge * 0.20   # 20% of monthly charge ~$13

# Define the four outcomes
benefit_TP = CLV - retention_cost   # Save customer, minus cost of offer
cost_FN    = -CLV                   # Lost customer — full CLV lost
cost_FP    = -retention_cost        # Wasted retention offer
benefit_TN = 0                      # No action needed — no cost

print('=== Cost-Profit Matrix ===')
print(f'  Average Monthly Charge  : ${avg_monthly_charge:.2f}')
print(f'  Customer Lifetime Value : ${CLV:.2f}  (12 months)')
print(f'  Retention Offer Cost    : ${retention_cost:.2f}  (20% of monthly charge)')
print()
print(f'  True Positive  (TP) benefit : +${benefit_TP:.2f}  (customer saved)')
print(f'  False Negative (FN) cost    : -${abs(cost_FN):.2f}  (customer lost)')
print(f'  False Positive (FP) cost    : -${abs(cost_FP):.2f}  (wasted offer)')
print(f'  True Negative  (TN) benefit :  ${benefit_TN:.2f}  (no action needed)')

### 7.2 — Visualise the Cost Matrix

In [ ]:
matrix_vals   = np.array([[benefit_TN, cost_FN],
                           [cost_FP,   benefit_TP]])
matrix_labels = np.array([['TN\n$0', f'FN\n-${abs(cost_FN):.0f}'],
                           [f'FP\n-${abs(cost_FP):.0f}', f'TP\n+${benefit_TP:.0f}']])

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(matrix_vals, annot=matrix_labels, fmt='',
            cmap='RdYlGn', ax=ax, linewidths=2, linecolor='white',
            xticklabels=['Actual: No Churn', 'Actual: Churn'],
            yticklabels=['Predicted: No Churn', 'Predicted: Churn'],
            annot_kws={'size': 14, 'weight': 'bold'},
            vmin=-CLV, vmax=CLV)

ax.set_title('Cost-Profit Matrix\n($ per customer, based on CLV assumptions)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_16_cost_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved ✓')

### 7.3 — Compute Total Profit at Default Threshold (0.5)

In [ ]:
profit_results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    cm     = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    total_profit = (tp * benefit_TP) + (fn * cost_FN) + \
                   (fp * cost_FP)    + (tn * benefit_TN)
    profit_per_customer = total_profit / len(y_test)

    profit_results.append({
        'Model'               : name,
        'TP'                  : tp,
        'FN'                  : fn,
        'FP'                  : fp,
        'TN'                  : tn,
        'Total Profit ($)'    : round(total_profit, 2),
        'Profit/Customer ($)' : round(profit_per_customer, 2)
    })

profit_df = pd.DataFrame(profit_results).set_index('Model')

print('=== Profit at Default Threshold (0.5) ===')
print(profit_df.to_string())

### 7.4 — Expected Maximum Profit (EMP) — Sweep All Thresholds

In [ ]:
# Instead of fixing threshold at 0.5, we find the threshold
# that MAXIMISES profit for each model — this is the EMP approach

thresholds   = np.linspace(0.01, 0.99, 200)
emp_results  = {}
profit_curves = {}

for name, model in models.items():
    y_proba  = model.predict_proba(X_test)[:, 1]
    profits  = []

    for thresh in thresholds:
        y_pred = (y_proba >= thresh).astype(int)
        cm     = confusion_matrix(y_test, y_pred)

        # Handle edge case where all predictions are one class
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
        else:
            tn, fp, fn, tp = len(y_test), 0, 0, 0

        profit = (tp * benefit_TP) + (fn * cost_FN) + \
                 (fp * cost_FP)    + (tn * benefit_TN)
        profits.append(profit)

    profit_curves[name]  = profits
    best_idx             = np.argmax(profits)
    emp_results[name]    = {
        'EMP ($)'               : round(profits[best_idx], 2),
        'EMP / Customer ($)'    : round(profits[best_idx] / len(y_test), 2),
        'Optimal Threshold'     : round(thresholds[best_idx], 3)
    }

emp_df = pd.DataFrame(emp_results).T
print('=== Expected Maximum Profit (EMP) ===')
print(emp_df.to_string())

### 7.5 — Plot Profit Curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors  = ['#2E86AB', '#E74C3C', '#2ECC71']

for (name, profits), color in zip(profit_curves.items(), colors):
    ax.plot(thresholds, profits, label=name, color=color, linewidth=2.5)

    # Mark the maximum profit point
    best_idx = np.argmax(profits)
    ax.scatter(thresholds[best_idx], profits[best_idx],
               color=color, s=120, zorder=5)
    ax.annotate(f'EMP=${profits[best_idx]:,.0f}\n@t={thresholds[best_idx]:.2f}',
                xy=(thresholds[best_idx], profits[best_idx]),
                xytext=(thresholds[best_idx] + 0.05, profits[best_idx] - 8000),
                fontsize=9, color=color, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5))

# Default threshold line
ax.axvline(0.5, color='gray', linestyle='--', linewidth=1.5,
           label='Default threshold (0.5)')
ax.axhline(0,   color='black', linestyle='-',  linewidth=0.8)

ax.set_title('Profit Curve vs Classification Threshold\n(Each point = total profit if that threshold is used)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Classification Threshold', fontsize=12)
ax.set_ylabel('Total Profit ($)', fontsize=12)
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
ax.legend(fontsize=10)
ax.set_xlim([0, 1])

plt.tight_layout()
plt.savefig('plot_17_profit_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved ✓')

### 7.6 — EMP Bar Chart

In [ ]:
emp_vals   = [emp_results[n]['EMP ($)'] for n in models]
model_names = list(models.keys())
colors     = ['#2E86AB', '#E74C3C', '#2ECC71']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(model_names, emp_vals, color=colors, edgecolor='white', width=0.5)

for bar, val in zip(bars, emp_vals):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 500,
            f'${val:,.0f}', ha='center', fontweight='bold', fontsize=12)

ax.set_title('Expected Maximum Profit (EMP) — Per Model',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Maximum Profit ($)')
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
plt.tight_layout()
plt.savefig('plot_18_emp_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved ✓')

### 7.7 — THE KEY FINDING: Traditional Ranking vs Profit Ranking

In [ ]:
# Traditional ranking (by ROC-AUC)
trad_rank = traditional_metrics['ROC-AUC'].rank(ascending=False).astype(int)

# Profit ranking (by EMP)
emp_series   = pd.Series({n: emp_results[n]['EMP ($)'] for n in models})
profit_rank  = emp_series.rank(ascending=False).astype(int)

comparison_df = pd.DataFrame({
    'ROC-AUC Score (%)' : traditional_metrics['ROC-AUC'],
    'Traditional Rank'  : trad_rank,
    'EMP ($)'           : emp_series.round(2),
    'Profit Rank'       : profit_rank,
    'Rank Changed?'     : (trad_rank != profit_rank).map({True: '⚠ YES', False: 'No'})
})

print('=== TRADITIONAL vs PROFIT-BASED RANKING ===')
print(comparison_df.to_string())

changed = comparison_df[comparison_df['Rank Changed?'] == '⚠ YES']
print(f'\n{len(changed)} model(s) changed rank between traditional and profit evaluation.')
print('\nThis is the core finding of the project:')
print('A model ranked higher by AUC does NOT necessarily generate more profit.')

comparison_df.to_csv('final_comparison.csv')
print('\nSaved: final_comparison.csv ✓')

### 7.8 — Side-by-Side Ranking Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors    = ['gold', 'silver', '#cd7f32']  # gold, silver, bronze

# Traditional ranking
trad_sorted = traditional_metrics['ROC-AUC'].sort_values(ascending=False)
axes[0].bar(range(len(trad_sorted)), trad_sorted.values,
            color=colors[:len(trad_sorted)], edgecolor='white', width=0.5)
axes[0].set_xticks(range(len(trad_sorted)))
axes[0].set_xticklabels([f'#{i+1}\n{n}' for i, n in enumerate(trad_sorted.index)],
                         fontsize=10)
for i, val in enumerate(trad_sorted.values):
    axes[0].text(i, val + 0.2, f'{val:.1f}%', ha='center', fontweight='bold')
axes[0].set_title('Ranking by ROC-AUC\n(Traditional)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('ROC-AUC (%)')
axes[0].set_ylim(70, 100)

# Profit ranking
profit_sorted = emp_series.sort_values(ascending=False)
axes[1].bar(range(len(profit_sorted)), profit_sorted.values,
            color=colors[:len(profit_sorted)], edgecolor='white', width=0.5)
axes[1].set_xticks(range(len(profit_sorted)))
axes[1].set_xticklabels([f'#{i+1}\n{n}' for i, n in enumerate(profit_sorted.index)],
                         fontsize=10)
for i, val in enumerate(profit_sorted.values):
    axes[1].text(i, val + 500, f'${val:,.0f}', ha='center', fontweight='bold')
axes[1].set_title('Ranking by EMP\n(Profit-Based)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Expected Maximum Profit ($)')
axes[1].yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))

plt.suptitle('Traditional Ranking  vs  Profit-Based Ranking',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot_19_ranking_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved ✓')

In [ ]:
print('='*60)
print('         STEP 7 COMPLETE — PROFIT EVALUATION SUMMARY')
print('='*60)
print(f'  Cost framework  : CLV-based (${CLV:.0f} per customer)')
print(f'  Retention cost  : ${retention_cost:.0f} per offer')
print(f'  Threshold sweep : 200 points from 0.01 to 0.99')
print()
for name in models:
    e = emp_results[name]
    print(f'  {name}')
    print(f'    EMP            : ${e["EMP ($)"]:,.2f}')
    print(f'    Per Customer   : ${e["EMP / Customer ($)"]:.2f}')
    print(f'    Best Threshold : {e["Optimal Threshold"]}')
print('='*60)
print('Step 7 Complete ✓  →  Proceed to Step 8: Conclusion')